In [ ]:
"""Document loading module for RAG system.

This module provides functionality to load FAQ entry data from JSON files
and convert it into formats suitable for RAG processing, including pandas
DataFrames and LangChain Document objects.
"""

from typing import List
from pathlib import Path
import json

import pandas as pd
from langchain_core.documents import Document


class DocumentLoader:
    """
    A utility class for loading and converting JSON FAQ entry data into LangChain Document objects.

    Input Requirement:
    - The input must be a JSON file with a list of FAQ entries.
    - Each entry must contain the following fields:
        - 'id': Unique identifier for each entry.
        - 'question': The FAQ question.
        - 'answer': The full answer text.
        - 'topic': FAQ topic/category (e.g., 'account', 'features', 'billing', 'troubleshooting').
        - 'last_updated': Timestamp indicating when the entry was last updated.

    Preferred date format for 'last_updated': 'YYYY-MM-DD' or 'YYYY-MM-DD HH:MM:SS'.

    Example JSON format:
    [
        {
            "id": "FAQ001",
            "question": "How do I reset my password?",
            "answer": "To reset your password, click 'Forgot Password'...",
            "topic": "account",
            "last_updated": "2024-10-05"
        }
    ]
    """

    def __init__(self, json_file: str):
        """
        Initializes the DocumentLoader with the path to a JSON file.

        Args:
            json_file (str): Absolute or relative path to the JSON file.

        Raises:
            ValueError: If input is not a non-empty string.
        """
        if not isinstance(json_file, str) or not json_file.strip():
            raise ValueError("json_file must be a non-empty string")
        self.json_file = json_file

    def load_data(self) -> pd.DataFrame:
        """
        Loads and parses FAQ entry data from the specified JSON file.

        Functionality:
        - Verifies that the file exists and is accessible.
        - Parses the JSON content into a pandas DataFrame.
        - Validates that each entry contains all required fields.
        - Returns a DataFrame with all entry data.

        Returns:
            pd.DataFrame: A DataFrame containing all FAQ entries with required columns:
                - 'id': Entry identifier
                - 'question': FAQ question
                - 'answer': FAQ answer
                - 'topic': Entry topic
                - 'last_updated': Last update date

        Raises:
            FileNotFoundError: If the file path does not exist.
            json.JSONDecodeError: If file contains invalid JSON.
            ValueError: If required fields are missing or DataFrame is empty.
            KeyError: If required columns are not present in the data.
        """
        file_path = Path(self.json_file)

        if not file_path.exists():
            raise FileNotFoundError(f"File not found: {self.json_file}")

        try:
            with file_path.open("r", encoding="utf-8") as f:
                json_data = json.load(f)
        except json.JSONDecodeError as err:
            raise json.JSONDecodeError(
                f"Invalid JSON in '{self.json_file}': {err}",
                doc=err.doc,
                pos=err.pos
            ) from err

        if not isinstance(json_data, list):
            raise ValueError("JSON root must be a list of FAQ entries.")

        data = pd.DataFrame(json_data)

        if data.empty:
            raise ValueError(f"no entries found in '{self.json_file}'")

        required_columns = {
            "id",
            "question",
            "answer",
            "topic",
            "last_updated",
        }

        missing_columns = required_columns - set(data.columns)
        if missing_columns:
            raise ValueError(f"Missing columns in '{self.json_file}': {', '.join(sorted(missing_columns))}")

        if any(data[col].isnull().any() for col in required_columns):
            raise ValueError(f"missing values found in required columns of '{self.json_file}'")

        return data

    def create_documents(self, data: pd.DataFrame) -> List[Document]:
        """
        Converts validated entry data from a DataFrame into LangChain Document objects.

        Args:
            data (pandas.DataFrame): A DataFrame where each row represents a FAQ entry. Required columns:
                - 'id'
                - 'question'
                - 'answer'
                - 'topic'
                - 'last_updated'

        Returns:
            List[Document]: A list of LangChain-compatible Document objects with metadata.
                Each Document should have:
                - page_content: The FAQ question + answer
                - metadata: Dictionary containing 'id', 'question', 'topic', 'last_updated'

        Raises:
            ValueError: If required columns are missing or DataFrame is empty.
            TypeError: If input is not a pandas DataFrame.
        """
        if not isinstance(data, pd.DataFrame):
            raise TypeError("Not a dataframe.")

        if data.empty:
            raise ValueError("DataFrame is empty.")

        required_columns = {
            "id",
            "question",
            "answer",
            "topic",
            "last_updated",
        }

        missing_columns = required_columns - set(data.columns)
        if missing_columns:
            raise ValueError(f"Missing columns in DataFrame: {', '.join(sorted(missing_columns))}")

        return [Document(
            page_content=row['question'] + " " + row['answer'],
            metadata={
                "id": row['id'],
                "question": row['question'],
                "topic": row['topic'],
                "last_updated": row['last_updated'],
            }
        ) for _, row in data.iterrows()]


In [ ]:
"""Vector store module for semantic search in RAG system.

This module provides functionality to create, manage, and query vector embeddings
for semantic search. It handles embedding generation, index creation/loading,
and similarity search operations using sentence transformers.
"""

from typing import Dict, List, Tuple

from pathlib import Path
import pickle

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer


class VectorStore:
    """
    A class for creating and managing vector embeddings and indexes for semantic search.

    Responsibilities:
    - Generate embeddings from FAQ questions using SentenceTransformer models.
    - Build and persist vector indexes with metadata.
    - Load pre-built indexes from disk.
    - Perform efficient similarity search using cosine similarity.
    - Manage document metadata alongside embeddings.

    **Expected Fields in DataFrame:**
    The `df` parameter passed to `create_index()` should include:
    - 'id': Unique entry identifier.
    - 'question': The FAQ question (used for embeddings).
    - 'answer': The answer text.
    - 'topic': Entry topic (e.g., 'account', 'features', 'billing').
    - 'last_updated': Timestamp of last update.

    **Embedding Generation:**
    Uses SentenceTransformer model `all-MiniLM-L6-v2` (384 dimensions) for embeddings.
    Embeddings should be generated from FAQ questions (as questions are more representative than answers).

    **Similarity Search:**
    Uses cosine similarity to find the most relevant documents for a query.
    Implements brute-force approach: compute similarity with all vectors, then return top-k.
    """

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initializes the VectorStore with a specified embedding model.

        Args:
            model_name (str): Name of the SentenceTransformer model to use.
                Default: "all-MiniLM-L6-v2" (384 dimensions)
        """
        self.model_name = model_name
        self.model = SentenceTransformer(model_name)

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generates sentence embeddings for a list of texts using SentenceTransformer.

        Args:
            texts (List[str]): A list of text strings (e.g., FAQ questions).

        Returns:
            np.ndarray: Array of sentence embedding vectors (shape: [n_texts, embedding_dim]).
                For all-MiniLM-L6-v2, embedding_dim = 384.

        Raises:
            ValueError: If `texts` is not a non-empty list.
            TypeError: If `texts` is not a list.
        """
        if not isinstance(texts, list):
            raise TypeError("Input must be a list of strings.")

        if not texts:
            raise ValueError("Input list is empty.")

        if not all(isinstance(text, str) for text in texts):
            raise TypeError("All elements in the input list must be strings.")

        try:
            return self.model.encode(
                texts,
                convert_to_numpy=True,
                show_progress_bar=False,
            )
        except Exception as exc:
            raise RuntimeError("Failed to generate embeddings.") from exc

    def create_index(self, df: pd.DataFrame, index_file_name: str, index_folder_name: str) -> Dict:
        """
        Creates a vector index from the given DataFrame and saves it to disk.

        Process:
        1. Generate embeddings for FAQ questions.
        2. Create index dictionary with embeddings and metadata.
        3. Save index to disk using pickle.
        4. Return the created index.

        Args:
            df (pd.DataFrame): DataFrame containing FAQ entries with required fields:
                - 'id': Unique entry ID.
                - 'question': FAQ question (used for embeddings).
                - 'answer': FAQ answer.
                - 'topic': Entry topic.
                - 'last_updated': Last updated timestamp.
            index_file_name (str): Name of the index file (e.g., 'faq_index.pkl').
            index_folder_name (str): Directory where the index will be saved.

        Returns:
            dict: The created index containing:
                - 'embeddings': numpy array of embeddings (shape: [n_docs, 384])
                - 'id': list of entry IDs
                - 'question': list of questions
                - 'answer': list of answers
                - 'topic': list of topics
                - 'last_updated': list of last updated timestamps

        Raises:
            ValueError: If DataFrame is empty or missing required columns.
            OSError: If unable to create directory or save file.
        """
        required_columns = {
            "id",
            "question",
            "answer",
            "topic",
            "last_updated",
        }

        missing_columns = required_columns - set(df.columns)
        if missing_columns:
            raise ValueError(f"Missing columns in DataFrame: {', '.join(missing_columns)}")

        # Create embeddings for FAQ questions
        df['embedding'] = list(self.generate_embeddings(df['question'].tolist()))

        # Create index dictionary
        index = {
            "embeddings": np.array(df['embedding'].tolist()),
            "id": df['id'].tolist(),
            "question": df['question'].tolist(),
            "answer": df['answer'].tolist(),
            "topic": df['topic'].tolist(),
            "last_updated": df['last_updated'].tolist(),
        }

        # Save index to disk
        index_path = Path(index_folder_name) / index_file_name

        if not index_path.parent.exists():
            index_path.parent.mkdir(parents=True)

        with open(index_path, 'wb') as f:
            pickle.dump(index, f)

        return index

    def load_index(self, index_file_name: str, index_folder_name: str) -> Dict:
        """
        Loads a precomputed vector index from disk.

        Args:
            index_file_name (str): The file name of the saved index (e.g., 'faq_index.pkl').
            index_folder_name (str): The directory where the index is stored.

        Returns:
            dict: The loaded index containing embeddings and metadata (same structure as create_index).

        Raises:
            FileNotFoundError: If the index file does not exist.
            ValueError: If the index structure is invalid or corrupted.
            KeyError: If required keys are missing in the index.
        """
        index_path = Path(index_folder_name) / index_file_name

        if not index_path.is_file():
            raise FileNotFoundError(f"Index file not found: {index_path}")

        try:
            with index_path.open("rb") as f:
                index = pickle.load(f)
        except (
            pickle.UnpicklingError,
            EOFError,
            AttributeError,
            ImportError,
            IndexError,
        ) as exc:
            raise ValueError(f"Invalid or corrupted index file: {index_path}") from exc

        if not isinstance(index, dict):
            raise ValueError("Loaded index must be a dictionary.")

        required_keys = {
            "embeddings",
            "id",
            "question",
            "answer",
            "topic",
            "last_updated",
        }

        missing_keys = required_keys - index.keys()
        if missing_keys:
            raise KeyError(
                f"Missing required keys: {', '.join(sorted(missing_keys))}"
            )

        if not isinstance(index["embeddings"], np.ndarray):
            raise ValueError("'embeddings' must be a numpy.ndarray.")

        n = len(index["embeddings"])

        for key in ("id", "question", "answer", "topic", "last_updated"):
            if len(index[key]) != n:
                raise ValueError(
                    f"Length mismatch: '{key}' contains {len(index[key])} items, "
                    f"expected {n}."
                )

        return index

    def get_query_embedding(self, query: str) -> np.ndarray:
        """
        Generates an embedding for a single query string.

        Args:
            query (str): The query string to embed.

        Returns:
            np.ndarray: The embedding vector for the query (shape: [embedding_dim]).
                For all-MiniLM-L6-v2, shape = (384,).

        Raises:
            ValueError: If `query` is not a valid non-empty string.
        """
        if not query:
            raise ValueError("Invalid query: Query must be a non-empty string.")

        try:
            query_embedding = self.model.encode(query, convert_to_numpy=True)
        except Exception as e:
            raise RuntimeError(f"Error generating query embedding: {e}")

        return query_embedding

    def find_top_k_matches(self, query_embedding: np.ndarray, index: Dict, k: int = 5) -> List[Tuple[int, float]]:
        """
        Finds the top-k most similar entries from the index using cosine similarity.

        Implementation should use brute-force approach:
        1. Compute cosine similarity between query_embedding and all embeddings in index
        2. Sort results by similarity score (descending)
        3. Return top-k matches

        Cosine similarity formula:
        similarity = dot(a, b) / (norm(a) * norm(b))

        Args:
            query_embedding (np.ndarray): The embedding of the input query (shape: [384]).
            index (dict): The index containing document embeddings and metadata.
                Expected structure: {'embeddings': np.ndarray, 'id': list, ...}
            k (int): The number of top matches to return (default: 5).

        Returns:
            list: A list of tuples (document_index, similarity_score) representing the top-k matches,
                  sorted by similarity score in descending order.
                  - document_index: Index position in the original DataFrame/index
                  - similarity_score: Cosine similarity score (float, typically 0.0 to 1.0)

        Raises:
            ValueError: If inputs are invalid (e.g., index missing embeddings).
            TypeError: If query_embedding is not a numpy array.
        """
        if not isinstance(query_embedding, np.ndarray):
            raise TypeError("Invalid query_embedding: Must be a numpy array.")

        if "embeddings" not in index:
            raise ValueError("Invalid index: Missing 'embeddings'.")

        if len(query_embedding) != index["embeddings"].shape[1]:
            raise ValueError("Dimension mismatch.")

        similarities = []
        for i, doc_embedding in enumerate(index["embeddings"]):
            similarity = np.dot(query_embedding, doc_embedding) / (
                np.linalg.norm(query_embedding) * np.linalg.norm(doc_embedding)
            )
            similarities.append((i, similarity))

        # Sort by similarity score (descending) and return top-k
        similarities.sort(key=lambda x: x[1], reverse=True)
        return similarities[:k]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3517.65it/s]


[-6.95707500e-02  9.52000022e-02  1.60214063e-02  6.80148089e-03
 -8.84049684e-02  1.42048411e-02  5.40277548e-02  4.56368662e-02
 -3.19217592e-02 -2.95636989e-02  5.67037053e-03  6.36050338e-03
  2.30016634e-02 -5.11627272e-02 -4.39477041e-02  5.89560904e-02
  5.50739579e-02  9.31472704e-02 -1.99183058e-02  5.56543469e-03
 -6.06279187e-02  7.54946470e-02 -1.02012418e-03 -3.40628959e-02
  4.00799662e-02  4.76241112e-02 -3.31660584e-02 -7.22288736e-04
  5.98663352e-02 -8.33192188e-03 -9.59181972e-03  5.34926951e-02
 -2.39231512e-02 -2.01412346e-02 -4.85538989e-02  1.87074617e-02
 -2.20365413e-02  6.23629317e-02  4.00254037e-03  2.97090597e-02
  1.27504906e-02  1.79136044e-03 -2.17953082e-02 -8.11001286e-02
 -2.61442736e-04  3.26574743e-02 -1.05920080e-02  3.05196345e-02
  3.11776660e-02 -1.16895279e-02  5.91250323e-03  5.38987340e-03
  2.91767698e-02  5.12313731e-02 -3.31498981e-02 -9.99479070e-02
  7.15584978e-02 -6.35875240e-02 -3.19283791e-02  2.67297886e-02
 -4.80684936e-02 -2.02164